## `01_code_interact_basic.py`

This script demonstrates basic usage of Python's `code` module, particularly `code.interact()`, to drop into an interactive REPL within a running program.

In [ ]:
"""
Session 3 - Demo 1: code.interact() basics
Run this and interact with the shell to inspect variables.
Exit with Ctrl+D to continue, or exit() to terminate.
"""
import code

x = 42
y = "Hello TSAI"
z = [1, 2, 3, 4, 5]

print("Variables defined: x, y, z")
print("Dropping into interactive shell...")
print("Try typing: x, y, z, type(x), len(z)")
print("Exit with Ctrl+D to continue, exit() to stop\n")

# Drop into interactive shell
code.interact(local=locals())

print("\nYou continued! This line runs because you used Ctrl+D.")
print(f"x is still {x}, y is still {y}")


## `02_code_interact_agent.py`

This script shows how an AI agent can leverage an interactive Python environment to execute code and test logic dynamically.

In [ ]:
"""
Session 3 - Demo 2: code.interact() for debugging agents — GUIDED TOUR
=======================================================================

Run this and follow along. You'll be prompted to press Enter between steps
and given specific commands to try inside the interactive shell.

The goal: see WHY code.interact() is powerful for debugging agent loops.
"""
import code
import json
import time


# ─────────────────────────────────────────────────────────────
# Setup: a simulated agent (no real LLM, responses are hardcoded
# so the demo is deterministic and teaches the concept)
# ─────────────────────────────────────────────────────────────

SIMULATED_RESPONSES = [
    # Iteration 1: LLM decides to call the weather tool
    '{"tool_name": "get_weather", "tool_arguments": {"city": "Mumbai"}}',
    # Iteration 2: LLM has the weather, now gives the final answer
    '{"answer": "The weather in Mumbai is 32°C and humid. Yes, it is hotter than 30°C."}',
]
_response_index = 0


def fake_call_llm(conversation):
    """Pretends to call an LLM. Returns canned responses for the demo."""
    global _response_index
    resp = SIMULATED_RESPONSES[_response_index]
    _response_index += 1
    return resp


def get_weather(city: str) -> str:
    return json.dumps({"weather": "32°C, Humid, Partly Cloudy"})


tools = {"get_weather": get_weather}


# ─────────────────────────────────────────────────────────────
# Helpers for the guided tour
# ─────────────────────────────────────────────────────────────

def pause(message="Press ENTER to continue..."):
    """Wait for user to press enter. Makes the demo interactive."""
    input(f"\n  {message}")


def banner(text, char="="):
    print()
    print(char * 64)
    print(f"  {text}")
    print(char * 64)


def narrator(text):
    """Print a narration block with an arrow to distinguish from agent output."""
    print()
    for line in text.strip().split("\n"):
        print(f"  → {line}")


# ─────────────────────────────────────────────────────────────
# The guided agent loop
# ─────────────────────────────────────────────────────────────

def guided_agent_loop(user_query):
    banner(f"GUIDED TOUR: Debugging an Agent with code.interact()")

    narrator(f"""
This is a SIMULATED agent. The LLM responses are hardcoded so we can
focus on ONE thing: how code.interact() lets you peek inside the loop.

User's query: "{user_query}"

The agent will:
  1. Ask the LLM what to do
  2. Parse the response
  3. Call a tool if needed
  4. Repeat until it has a final answer

At key moments, we'll FREEZE execution and drop into a Python shell
so you can inspect the program's state with your own eyes.
""")

    pause("Press ENTER to start the agent...")

    # Initial conversation state
    conversation = [{"role": "user", "content": user_query}]

    for iteration in range(5):
        banner(f"ITERATION {iteration + 1}", char="─")

        narrator("About to call the LLM. The LLM will see the full conversation\n"
                 "history so far and decide what to do next.")

        pause("Press ENTER to call the LLM...")

        llm_response = fake_call_llm(conversation)

        print(f"\n  LLM returned:  {llm_response}")

        # ─────────────────────────────────────────────
        # BREAKPOINT: freeze and let user inspect
        # ─────────────────────────────────────────────
        narrator(f"""
We're about to process this response. But wait — in REAL agent development,
this is where things go wrong. The LLM might return broken JSON, the wrong
tool name, or weird arguments.

Let's FREEZE the program right here and poke around. I'm going to drop you
into a live Python shell where EVERY variable in scope is available.

Try typing these commands one at a time:

    >>> llm_response
              ...the raw string the LLM returned

    >>> iteration
              ...which loop iteration we're on

    >>> conversation
              ...the full message history so far

    >>> len(conversation)
              ...how many messages deep we are

    >>> tools
              ...what tools this agent can call

    >>> tools["get_weather"]("Delhi")
              ...you can even CALL a tool manually from the shell!

When you're done exploring, press Ctrl+D to exit the shell and continue.
""")

        pause("Press ENTER to drop into the interactive shell...")

        print()
        print("  ┌─────────────────────────────────────────────────────┐")
        print("  │  You're now in the interactive shell (Ctrl+D to exit) │")
        print("  └─────────────────────────────────────────────────────┘")

        code.interact(banner="", local=locals())

        print()
        print("  ┌─────────────────────────────────────────────────────┐")
        print("  │  Back in the agent loop.                            │")
        print("  └─────────────────────────────────────────────────────┘")

        # ─────────────────────────────────────────────
        # Now actually process the LLM response
        # ─────────────────────────────────────────────

        parsed = json.loads(llm_response)

        if "answer" in parsed:
            narrator(f"The LLM gave a FINAL ANSWER. The loop ends here.")
            print(f"\n  FINAL ANSWER:  {parsed['answer']}")
            break

        if "tool_name" in parsed:
            tool_name = parsed["tool_name"]
            tool_args = parsed["tool_arguments"]

            narrator(f"The LLM wants to call tool '{tool_name}' with args {tool_args}.\n"
                     f"We'll execute it, then feed the result back to the LLM.")

            pause("Press ENTER to execute the tool...")

            result = tools[tool_name](**tool_args)
            print(f"\n  Tool returned:  {result}")

            conversation.append({"role": "assistant", "content": llm_response})
            conversation.append({"role": "tool", "content": result})

            narrator(f"Added the tool call and result to conversation history.\n"
                     f"conversation is now {len(conversation)} messages long.\n"
                     f"In iteration {iteration + 2}, the LLM will see ALL of this.")


def summary():
    banner("WHAT YOU JUST SAW", char="=")
    narrator("""
You used code.interact() to freeze a running agent and inspect its state.
This is how you debug real agents in production:

  - Your agent gets stuck? Drop code.interact() into the loop.
  - Wrong tool being called? Drop code.interact() BEFORE the tool call.
  - Conversation getting too long? Print len(conversation) in the shell.
  - Want to test a tool manually? Call it from the shell.

code.interact() is your emergency pause button for agents. Use it.

For structured step-by-step debugging (next breakpoint, step into, etc.),
use pdb instead — see 03_pdb_basic.py.
""")


if __name__ == "__main__":
    guided_agent_loop("What is the weather in Mumbai? Is it hotter than 30 degrees?")
    summary()


## `03_pdb_basic.py`

An introduction to Python's built-in debugger (`pdb`). It covers how to set tracepoints and step through code interactively.

In [ ]:
"""
Session 3 - Demo 3: pdb basics
Run this and use pdb commands:
  p x    - print x
  p y    - print y
  s      - step into the add() function
  n      - next line (don't step into functions)
  c      - continue execution
  b 11   - set breakpoint at line 11
  q      - quit
"""

def add(a, b):
    result = a + b
    return result

x = 5
y = 15
breakpoint()  # Modern way (Python 3.7+) — same as: from pdb import set_trace; set_trace()

z = add(x, y)
print("Result:", z)


## `04_async_blocking.py`

This example highlights what happens when blocking synchronous operations (like `time.sleep()`) are incorrectly used inside an `async` function, preventing other tasks from running.

In [ ]:
"""
Session 3 - Demo 4: Blocking (synchronous) code
Run this first, then run 05_async_nonblocking.py to see the difference.
"""
import time

def say_hello():
    time.sleep(2)
    print("Hello World!")

def say_good_bye():
    time.sleep(2)
    print("GoodBye World!")

start = time.time()
say_hello()
say_good_bye()
total = time.time() - start
print(f"Total time for BLOCKING version: {total:.2f} seconds")
print("(Each function waited for the other to finish)")


## `05_async_nonblocking.py`

A corrected version of the previous script, demonstrating proper asynchronous programming using non-blocking calls (like `asyncio.sleep()`) to allow concurrent task execution.

In [ ]:
"""
Session 3 - Demo 5: Non-blocking (asynchronous) code
Compare the time with 04_async_blocking.py!
"""
import asyncio
import time

async def say_hello():
    await asyncio.sleep(2)  # Non-blocking sleep
    print("Hello World!")

async def say_good_bye():
    await asyncio.sleep(2)  # Non-blocking sleep
    print("GoodBye World!")

async def main():
    start = time.time()

    # Run both functions concurrently
    await asyncio.gather(say_hello(), say_good_bye())

    total = time.time() - start
    print(f"Total time for NON-BLOCKING version: {total:.2f} seconds")
    print("(Both functions ran concurrently!)")

# Run the async program
asyncio.run(main())


## `06_async_common_mistake.py`

This script explores common pitfalls when working with Python's `asyncio`, such as forgetting to `await` coroutines or blocking the event loop.

In [ ]:
"""
Session 3 - Demo 6: The async mistake you WILL make
This shows what happens when you forget 'await'.
"""
import asyncio

async def call_llm(prompt: str) -> str:
    """Simulates an async LLM call"""
    await asyncio.sleep(1)  # Simulating network delay
    return f"LLM response to: {prompt}"

async def main():
    # WRONG - forgetting await
    result_wrong = call_llm("Hello")
    print(f"Without await: {result_wrong}")
    print(f"Type: {type(result_wrong)}")
    # This prints: <coroutine object call_llm at 0x...>
    # NOT the actual response!

    print()

    # RIGHT - using await
    result_right = await call_llm("Hello")
    print(f"With await: {result_right}")
    print(f"Type: {type(result_right)}")
    # This prints: "LLM response to: Hello"
    # The ACTUAL response!

asyncio.run(main())


## `07_python_essentials.py`

A review of essential Python concepts and syntax, acting as a foundational primer for the subsequent agent-based scripts.

In [ ]:
"""
Session 3 - Demo 7: Python essentials for agentic systems
Covers: try/except, dataclasses, decorators, f-strings
"""
import json
from dataclasses import dataclass

# === try/except: Handling messy LLM responses ===

print("=" * 50)
print("1. try/except — Handling messy LLM responses")
print("=" * 50)

# LLMs sometimes wrap JSON in markdown code fences
messy_response = '```json\n{"tool": "calculate", "args": {"expr": "2+2"}}\n```'

print(f"Messy LLM response: {messy_response}")

# This will crash:
try:
    data = json.loads(messy_response)
except json.JSONDecodeError as e:
    print(f"Direct parse failed: {e}")

    # Clean it up and try again
    cleaned = messy_response.strip().strip('`').strip()
    if cleaned.startswith('json'):
        cleaned = cleaned[4:].strip()
    data = json.loads(cleaned)
    print(f"After cleaning: {data}")


# === dataclasses: Contracts between LLM and tools ===

print(f"\n{'=' * 50}")
print("2. dataclasses — Contracts between LLM and tools")
print("=" * 50)

@dataclass
class ToolCall:
    name: str
    arguments: dict

@dataclass
class AgentResponse:
    tool_call: ToolCall | None = None
    final_answer: str | None = None

# Creating a tool call response
response = AgentResponse(
    tool_call=ToolCall(name="calculate", arguments={"expression": "2**10"})
)
print(f"Tool call: {response.tool_call.name}({response.tool_call.arguments})")

# Creating a final answer response
response2 = AgentResponse(final_answer="The result is 1024.")
print(f"Final answer: {response2.final_answer}")


# === Decorators: Tool registration ===

print(f"\n{'=' * 50}")
print("3. Decorators — Automatic tool registration")
print("=" * 50)

TOOLS = {}

def tool(func):
    """Decorator that registers a function as a tool"""
    TOOLS[func.__name__] = func
    return func

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression"""
    return str(eval(expression))

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city"""
    return f"Weather in {city}: 28°C, partly cloudy"

@tool
def reverse_text(text: str) -> str:
    """Reverse a string"""
    return text[::-1]

print(f"Registered tools: {list(TOOLS.keys())}")

# Call any tool by name — this is how agents dispatch tool calls
tool_name = "calculate"
tool_args = {"expression": "2**10"}
result = TOOLS[tool_name](**tool_args)
print(f"Called {tool_name}({tool_args}) → {result}")

tool_name = "reverse_text"
tool_args = {"text": "TSAI EAG V3"}
result = TOOLS[tool_name](**tool_args)
print(f"Called {tool_name}({tool_args}) → {result}")


# === f-strings: Building system prompts dynamically ===

print(f"\n{'=' * 50}")
print("4. f-strings — Dynamic system prompts")
print("=" * 50)

# Build tool descriptions dynamically from registered tools
tool_descriptions = []
for name, func in TOOLS.items():
    doc = func.__doc__ or "No description"
    tool_descriptions.append(f"  - {name}: {doc}")

tool_list = "\n".join(tool_descriptions)

system_prompt = f"""You are a helpful AI agent with access to these tools:

{tool_list}

Respond in JSON format with either a tool call or final answer."""

print(system_prompt)


## `08_llm_basic.py`

This script covers basic integration with an LLM (Large Language Model) API, showing how to send a simple prompt and receive a response.

In [ ]:
"""
Session 3 - Demo 8: Talk to an LLM
First step — just call the API and see what comes back.

Before running:
  pip install google-genai python-dotenv
  Create a .env file next to this script with:
    GEMINI_API_KEY=your-key-here
    GEMINI_MODEL=gemini-2.5-flash-lite
"""
from google import genai
import os
import time
from dotenv import load_dotenv

load_dotenv()  # reads .env in the current directory

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
THROTTLE_SECONDS = 10  # Wait before each LLM call to stay under free-tier RPM limits

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not set. Create a .env file with GEMINI_API_KEY=...")

client = genai.Client(api_key=GEMINI_API_KEY)


def ask(prompt: str) -> str:
    """Send a prompt to Gemini and return the text response.

    Sleeps for THROTTLE_SECONDS before each call to stay under the free-tier
    rate limit (Gemini 3.1 Flash Lite: 15 RPM, 500 RPD).
    """
    print(f"  [waiting {THROTTLE_SECONDS}s to respect rate limits...]", flush=True)
    time.sleep(THROTTLE_SECONDS)
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return response.text

print(f"Using model: {GEMINI_MODEL}\n")

# Test 1: Something it CAN do (sort of)
print("=" * 50)
print("Test 1: Math question")
print("=" * 50)
q1 = "What is 2 raised to the power of 10?"
print(f"Q: {q1}")
print(f"A: {ask(q1)}")
print("(Might be right, might be wrong — it's just predicting text)")

# Test 2: Something it CANNOT do
print(f"\n{'=' * 50}")
print("Test 2: Real-time data")
print("=" * 50)
q2 = "What is the current temperature in Mumbai right now?"
print(f"Q: {q2}")
print(f"A: {ask(q2)}")
print("(This is a GUESS. It has no idea what the actual temperature is.)")

# Test 3: Something it will hallucinate on
print(f"\n{'=' * 50}")
print("Test 3: Calculation it might get wrong")
print("=" * 50)
q3 = "Calculate the sum of exponential values (e^x) of the first 6 Fibonacci numbers (1,1,2,3,5,8). Give just the number."
print(f"Q: Sum of e^x for first 6 Fibonacci numbers")
print(f"A: {ask(q3)}")

import math
actual = sum(math.exp(x) for x in [1, 1, 2, 3, 5, 8])
print(f"\nActual answer: {actual:.4f}")
print("(Compare — did the LLM get it right?)")
print("\nThis is WHY we need agents. LLMs can reason but they can't reliably ACT.")
print("An agent would call a calculate() tool and get the exact answer.")


## `09_llm_with_system_prompt.py`

Building on the previous script, this example demonstrates how to use system prompts to define the behavior, persona, and constraints of the LLM.

In [ ]:
"""
Session 3 - Demo 9: Give the LLM a system prompt that makes it an agent
Watch how the same LLM behaves differently with the right instructions.

Before running:
  pip install google-genai python-dotenv
  Create a .env file next to this script with:
    GEMINI_API_KEY=your-key-here
    GEMINI_MODEL=gemini-2.5-flash-lite
"""
from google import genai
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
THROTTLE_SECONDS = 10  # Wait before each LLM call to stay under free-tier RPM limits

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not set. Create a .env file with GEMINI_API_KEY=...")

client = genai.Client(api_key=GEMINI_API_KEY)


def ask(prompt: str) -> str:
    """Send a prompt to Gemini and return the text response.

    Sleeps for THROTTLE_SECONDS before each call to stay under the free-tier
    rate limit (Gemini 3.1 Flash Lite: 15 RPM, 500 RPD).
    """
    print(f"  [waiting {THROTTLE_SECONDS}s to respect rate limits...]", flush=True)
    time.sleep(THROTTLE_SECONDS)
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return response.text

system_prompt = """You are a helpful AI agent that can use tools to answer questions.

You have access to the following tools:

1. calculate(expression: str) -> str
   Evaluate a mathematical expression. Example: calculate("2**10")

2. get_weather(city: str) -> str
   Get the current weather for a city. Example: get_weather("Mumbai")

3. search_notes(query: str) -> str
   Search through user's notes. Example: search_notes("meeting agenda")

You must respond in ONE of these two JSON formats:

If you need to use a tool:
{"tool_name": "<name>", "tool_arguments": {"<arg_name>": "<value>"}}

If you have the final answer:
{"answer": "<your final answer>"}

IMPORTANT: Respond with ONLY the JSON. No other text. No markdown. No code fences.
"""

test_queries = [
    "What is the weather in Mumbai?",
    "What is 2 raised to the power of 10?",
    "Do I have any notes about meetings?",
    "What is the capital of France?",  # No tool needed — should give direct answer
]

for query in test_queries:
    print(f"\n{'=' * 50}")
    print(f"User: {query}")
    print("=" * 50)

    response_text = ask(f"{system_prompt}\n\nUser: {query}")
    print(f"Raw response: {response_text}")

    try:
        parsed = json.loads(response_text.strip())
        print(f"Parsed: {json.dumps(parsed, indent=2)}")

        if "tool_name" in parsed:
            print(f"  → LLM wants to call: {parsed['tool_name']}({parsed.get('tool_arguments', {})})")
        elif "answer" in parsed:
            print(f"  → LLM has final answer: {parsed['answer']}")
    except json.JSONDecodeError:
        print(f"  → Failed to parse! LLM didn't follow instructions perfectly.")
        print(f"  → This is NORMAL. This is why we need robust parsing.")


## `10_full_agent.py`

A comprehensive implementation of an autonomous agent that can process inputs, make decisions, and possibly use tools to achieve a specific goal.

In [ ]:
"""
Session 3 - Demo 10: The Full Agent
This is the complete agent with the agentic loop.
Everything from the session comes together here.

Before running:
  pip install google-genai python-dotenv
  Create a .env file next to this script with:
    GEMINI_API_KEY=your-key-here
    GEMINI_MODEL=gemini-2.5-flash-lite
"""
from google import genai
import json
import re
import math
import os
import time
from dotenv import load_dotenv

# ============================================================
# Configuration
# ============================================================
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
THROTTLE_SECONDS = 10  # Wait before each LLM call to stay under free-tier RPM limits

if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY not set. Create a .env file with GEMINI_API_KEY=...")

client = genai.Client(api_key=GEMINI_API_KEY)


def call_llm(prompt: str) -> str:
    """Send a prompt to Gemini and return the text response.

    Sleeps for THROTTLE_SECONDS before each call to stay under the free-tier
    rate limit (Gemini 3.1 Flash Lite: 15 RPM, 500 RPD).
    """
    print(f"  [waiting {THROTTLE_SECONDS}s to respect rate limits...]", flush=True)
    time.sleep(THROTTLE_SECONDS)
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return response.text


# ============================================================
# System Prompt — This is what turns an LLM into an agent
# ============================================================
system_prompt = """You are a helpful AI agent that can use tools to answer questions accurately.

You have access to the following tools:

1. calculate(expression: str) -> str
   Evaluate a mathematical expression using Python syntax.
   Examples: calculate("2**10"), calculate("math.sqrt(144)"), calculate("sum(math.exp(x) for x in [1,1,2,3,5,8])")

2. get_weather(city: str) -> str
   Get the current weather for a city.
   Examples: get_weather("Mumbai"), get_weather("London")

3. search_notes(query: str) -> str
   Search through user's notes for relevant information.
   Examples: search_notes("meeting"), search_notes("project ideas")

You must respond in ONE of these two JSON formats:

If you need to use a tool:
{"tool_name": "<name>", "tool_arguments": {"<arg_name>": "<value>"}}

If you have the final answer:
{"answer": "<your final answer>"}

IMPORTANT RULES:
- Respond with ONLY the JSON. No other text. No markdown code fences.
- Use tools when you need real data or precise calculations.
- After receiving a tool result, either use another tool or provide your final answer.
- For complex calculations, break them down into steps if needed.
- ALWAYS use the calculate tool for math — do NOT try to compute in your head.
"""


# ============================================================
# Tools — The functions the agent can call
# ============================================================

def calculate(expression: str) -> str:
    """Evaluate a mathematical expression safely"""
    try:
        # Allow math functions in the expression
        allowed = {
            "math": math,
            "abs": abs,
            "round": round,
            "pow": pow,
            "sum": sum,
            "min": min,
            "max": max,
            "range": range,
            "list": list,
        }
        result = eval(expression, {"__builtins__": {}}, allowed)
        return json.dumps({"result": str(result)})
    except Exception as e:
        return json.dumps({"error": f"Calculation failed: {str(e)}"})


def get_weather(city: str) -> str:
    """Get current weather for a city (simulated)"""
    # In a real agent, this would call a weather API like OpenWeatherMap
    weather_data = {
        "Mumbai": {"temp": "32°C", "condition": "Humid, Partly Cloudy", "humidity": "78%"},
        "Delhi": {"temp": "28°C", "condition": "Clear Sky", "humidity": "45%"},
        "London": {"temp": "15°C", "condition": "Rainy", "humidity": "85%"},
        "New York": {"temp": "22°C", "condition": "Sunny", "humidity": "55%"},
        "Tokyo": {"temp": "26°C", "condition": "Windy", "humidity": "60%"},
        "San Francisco": {"temp": "18°C", "condition": "Foggy", "humidity": "72%"},
    }
    if city in weather_data:
        return json.dumps({"weather": weather_data[city]})
    return json.dumps({"error": f"Weather data not available for {city}"})


def search_notes(query: str) -> str:
    """Search through user's notes (simulated)"""
    notes = [
        {"title": "Meeting Agenda", "content": "Discuss Q3 targets, review agent architecture, plan MCP integration"},
        {"title": "Shopping List", "content": "Milk, eggs, bread, coffee, batteries"},
        {"title": "Project Ideas", "content": "Build a stock monitoring agent, voice-based assistant, browser automation tool"},
        {"title": "Travel Plans", "content": "Tokyo trip in December, need to book flights and hotel by November"},
        {"title": "Learning Notes", "content": "Finish transformer session, practice async Python, read MCP docs"},
    ]
    results = [
        n for n in notes
        if query.lower() in n["title"].lower() or query.lower() in n["content"].lower()
    ]
    if results:
        return json.dumps({"results": results})
    return json.dumps({"results": "No notes found matching your query"})


# Tool registry — maps tool names to functions
tools = {
    "calculate": calculate,
    "get_weather": get_weather,
    "search_notes": search_notes,
}


# ============================================================
# Response Parser — Handles messy LLM output
# ============================================================

def parse_llm_response(text: str) -> dict:
    """Parse the LLM's response, handling common formatting issues"""
    text = text.strip()

    # Remove markdown code fences if present
    if text.startswith("```"):
        lines = text.split("\n")
        # Remove first line (opening fence)
        lines = lines[1:]
        # Remove last line if it's a closing fence
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
        # Remove language identifier
        if text.startswith("json"):
            text = text[4:].strip()

    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to find JSON object in the text
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    raise ValueError(f"Could not parse LLM response: {text[:200]}")


# ============================================================
# The Agent Loop — This is where the magic happens
# ============================================================

def run_agent(user_query: str, max_iterations: int = 5, verbose: bool = True):
    """
    Run the agent loop:
    User query → LLM → [Tool call → Result → LLM]* → Final answer

    This is THE pattern. Everything else in this course builds on this loop.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"  User: {user_query}")
        print(f"{'='*60}")

    # Conversation history — this is the agent's "working memory"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1} ---")

        # Build prompt from message history
        # Each iteration, the LLM sees EVERYTHING that happened before
        prompt = ""
        for msg in messages:
            if msg["role"] == "system":
                prompt += msg["content"] + "\n\n"
            elif msg["role"] == "user":
                prompt += f"User: {msg['content']}\n\n"
            elif msg["role"] == "assistant":
                prompt += f"Assistant: {msg['content']}\n\n"
            elif msg["role"] == "tool":
                prompt += f"Tool Result: {msg['content']}\n\n"

        # Call the LLM
        response_text = call_llm(prompt)
        if verbose:
            print(f"LLM: {response_text.strip()}")

        # Parse the response
        try:
            parsed = parse_llm_response(response_text)
        except (ValueError, json.JSONDecodeError) as e:
            if verbose:
                print(f"Parse error: {e}")
                print("Asking LLM to retry...")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": "Please respond with valid JSON only. No markdown, no extra text."})
            continue

        # Check if it's a final answer
        if "answer" in parsed:
            if verbose:
                print(f"\n{'='*60}")
                print(f"  Agent Answer: {parsed['answer']}")
                print(f"{'='*60}")
            return parsed["answer"]

        # It's a tool call — execute it
        if "tool_name" in parsed:
            tool_name = parsed["tool_name"]
            tool_args = parsed.get("tool_arguments", {})

            if verbose:
                print(f"→ Calling tool: {tool_name}({tool_args})")

            # Check if tool exists
            if tool_name not in tools:
                error_msg = json.dumps({"error": f"Unknown tool: {tool_name}. Available: {list(tools.keys())}"})
                if verbose:
                    print(f"→ Error: {error_msg}")
                messages.append({"role": "assistant", "content": response_text})
                messages.append({"role": "tool", "content": error_msg})
                continue

            # Execute the tool
            tool_result = tools[tool_name](**tool_args)
            if verbose:
                print(f"→ Result: {tool_result}")

            # Add to conversation history — the LLM will see this next iteration
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "tool", "content": tool_result})

    print("\nMax iterations reached. Agent could not complete the task.")

    # Print full conversation for debugging
    if verbose:
        print(f"\n{'='*60}")
        print("Full conversation history:")
        print(f"{'='*60}")
        for i, msg in enumerate(messages):
            print(f"[{i}] {msg['role']}: {msg['content'][:100]}...")

    return None


# ============================================================
# Run the agent!
# ============================================================

if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("  SESSION 3: YOUR FIRST AI AGENT")
    print("  Let's see the agent loop in action!")
    print("=" * 60)

    # Test 1: Simple tool call
    print("\n\n>>> TEST 1: Simple weather query")
    run_agent("What is the weather in Mumbai?")

    # Test 2: Calculation
    print("\n\n>>> TEST 2: Math that LLMs get wrong")
    run_agent("What is 2 raised to the power of 10, plus the square root of 144?")

    # Test 3: Multi-step — needs multiple tool calls
    print("\n\n>>> TEST 3: Multi-step reasoning")
    run_agent(
        "Search my notes for travel plans, then check the weather in "
        "the city I'm planning to visit."
    )

    # Test 4: The assignment example!
    print("\n\n>>> TEST 4: The classic — sum of exponentials of Fibonacci")
    run_agent(
        "Calculate the sum of exponential values (e^x) of the "
        "first 6 Fibonacci numbers: 1, 1, 2, 3, 5, 8"
    )

    # Test 5: Something that doesn't need tools
    print("\n\n>>> TEST 5: No tools needed")
    run_agent("What is the capital of France?")


## `11_fake_agent.py`

A mock or 'fake' agent implementation, useful for testing interactions, interfaces, or pipelines without incurring the cost or latency of real LLM calls.

In [ ]:
"""
Session 3 - Demo 11: The Fake Agent (a.k.a. "Arcturus Jr.")
============================================================

This is NOT a real AI agent. There's no LLM here. It's a regex-based router
with a fake personality — but it looks and feels remarkably close to a real
agent because it:

  1. Routes queries to tools (just like a real agent)
  2. Calls real APIs over the internet
  3. Has conversational personality ("let me check that for you...")
  4. Handles small talk (how are you, what can you do, etc.)

The LESSON:
  A real AI agent is EXACTLY this structure — router + tools. The ONLY
  difference is the router. This fake agent uses regex. A real agent uses
  an LLM. Watch how this agent FAILS on queries slightly outside the
  regex patterns, and then compare with the LLM-powered agent (10_full_agent.py).

Before running:
  pip install requests
"""

import re
import random
import time
import json
import math
import datetime
import requests

# ============================================================
# The Agent's Persona
# ============================================================

AGENT_NAME = "Arcturus Jr."
AGENT_VERSION = "0.1 (no LLM, just regex + vibes)"

# Random filler phrases — makes responses feel alive instead of canned
FILLERS = {
    "thinking": [
        "Hmm, let me think...",
        "One moment...",
        "Give me a sec...",
        "Let me see...",
    ],
    "searching": [
        "Let me look that up for you.",
        "Checking the internet...",
        "Searching...",
        "On it!",
        "Looking that up now.",
    ],
    "calculating": [
        "Crunching the numbers...",
        "Let me calculate that...",
        "Working it out...",
    ],
    "got_it": [
        "Here you go:",
        "Got it!",
        "Here's what I found:",
        "Okay, here:",
        "Check this out:",
    ],
    "sorry_tool_failed": [
        "Hmm, that didn't work. The internet might be acting up.",
        "Sorry, I couldn't reach that service right now.",
        "Oof, something went wrong on my end.",
        "The API is being moody. Try again?",
    ],
}

def _say(key):
    """Pick a random filler phrase."""
    return random.choice(FILLERS[key])

# ============================================================
# Small helpers — make the agent feel alive
# ============================================================

def _type_out(text, delay=0.01):
    """Print text with a tiny delay between characters to feel 'alive'."""
    for char in text:
        print(char, end="", flush=True)
        time.sleep(delay)
    print()


def _think(message="thinking", seconds=1.2):
    """
    Show a spinning / blinking animation while the agent 'thinks'.
    Overwrites itself on the same line, then clears when done.
    """
    spinner = ["⠋", "⠙", "⠹", "⠸", "⠼", "⠴", "⠦", "⠧", "⠇", "⠏"]
    end = time.time() + seconds
    i = 0
    while time.time() < end:
        frame = spinner[i % len(spinner)]
        print(f"\r{AGENT_NAME}: {frame} {message}...   ", end="", flush=True)
        time.sleep(0.08)
        i += 1
    # Clear the line so the real response replaces the animation
    print(f"\r{' ' * 80}\r", end="", flush=True)


def _dots(message="one sec", seconds=1.0):
    """Simpler blinking dots animation — fallback if spinner looks weird."""
    end = time.time() + seconds
    dots = 0
    while time.time() < end:
        print(f"\r{AGENT_NAME}: {message}{'.' * (dots + 1)}   ", end="", flush=True)
        dots = (dots + 1) % 3
        time.sleep(0.35)
    print(f"\r{' ' * 80}\r", end="", flush=True)

def _reply(text, filler_key=None):
    """
    Return a formatted reply. If filler_key given, prepend a random filler.
    """
    if filler_key:
        return f"{_say(filler_key)}\n\n{text}"
    return text


# ============================================================
# SMALL TALK HANDLERS (no tools, just personality)
# ============================================================

def hello(match):
    greetings = [
        f"Hey there! I'm {AGENT_NAME}. What can I do for you?",
        f"Hi! {AGENT_NAME} here. How can I help?",
        f"Hello! I'm {AGENT_NAME}, nice to meet you. What do you need?",
    ]
    return random.choice(greetings)

def how_are_you(match):
    moods = [
        "I'm doing great, thanks for asking! Ready to help. What's on your mind?",
        "Pretty good for a regex-based agent! How about you?",
        "Feeling chatty today. What can I do for you?",
        "Functioning at 100% — no LLM, no problems. How about you?",
    ]
    return random.choice(moods)

def about_me(match):
    return (
        f"I'm {AGENT_NAME}, version {AGENT_VERSION}. "
        f"I'm a teaching assistant for EAG V3 Session 3. "
        f"I look like a real AI agent, but I'm actually just Python regex + some API calls. "
        f"My job is to show you what the 'skeleton' of an agent looks like before we add an LLM brain."
    )

def who_made_you(match):
    return (
        "I was built by Rohan Shravan for The School of AI. "
        "Actually, I was mostly built by Claude, but don't tell Rohan."
    )

def are_you_human(match):
    return (
        "Nope, I'm a program! And unlike ChatGPT, I'm not even an LLM — "
        "I'm literally just a pile of if-statements wearing a trench coat."
    )

def capabilities(match):
    return (
        "Here's what I can do:\n"
        "  - weather in <city>\n"
        "  - calculate <expression>  (or 'what is 2+2')\n"
        "  - what time is it / what's the date\n"
        "  - tell me about <topic>  (Wikipedia)\n"
        "  - define <word>\n"
        "  - tell me a joke\n"
        "  - cat fact / dog fact\n"
        "  - give me a quote\n"
        "  - search <query>  (DuckDuckGo)\n"
        "  - convert 100 USD to INR\n"
        "  - my ip / where am i\n"
        "  - random number between X and Y\n"
        "  - flip a coin / roll a die\n"
        "\n"
        "Try me! And when I fail, remember — THAT is why we need LLMs."
    )

def thanks(match):
    responses = [
        "You're welcome!",
        "Anytime!",
        "Glad I could help!",
        "No problem at all.",
        "Happy to help!",
    ]
    return random.choice(responses)

def compliment(match):
    return "Aww, thanks! You're pretty great yourself."

def bored(match):
    options = [
        "Want me to tell you a joke? Just say 'tell me a joke'.",
        "I can tell you a cat fact. Or give you an inspirational quote. Your call.",
        "I know — ask me something weird and let's see if my regex can handle it.",
    ]
    return random.choice(options)

def time_greeting(match):
    hour = datetime.datetime.now().hour
    if hour < 12:
        return "Good morning! What can I do for you today?"
    elif hour < 17:
        return "Good afternoon! How can I help?"
    else:
        return "Good evening! What's up?"

# Special marker returned when user wants to exit
GOODBYE = "__GOODBYE__"

def goodbye(match):
    farewells = [
        "Bye! Come back soon.",
        "See you later! Remember — I'm just regex. The LLM agent is cooler.",
        "Take care!",
        "Goodbye! Don't forget to run 10_full_agent.py to see the real thing.",
    ]
    print(f"\n{AGENT_NAME}: {random.choice(farewells)}\n")
    return GOODBYE


# ============================================================
# TOOL HANDLERS (these actually hit the internet)
# ============================================================

def _safe_get(url, timeout=10, **kwargs):
    """Wrapper around requests.get with friendly error handling."""
    try:
        return requests.get(url, timeout=timeout, **kwargs)
    except requests.RequestException:
        return None


def weather(match):
    city = match.group("city").strip().rstrip("?.").strip()
    _think(f"checking weather in {city.title()}", seconds=1.4)
    r = _safe_get(f"https://wttr.in/{city}?format=3")
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    return _reply(r.text.strip(), "got_it")


def calculate(match):
    expr = match.group("expr").strip()
    _think("crunching the numbers", seconds=0.8)
    try:
        allowed = {"math": math, "abs": abs, "round": round, "pow": pow,
                   "sum": sum, "min": min, "max": max, "pi": math.pi, "e": math.e}
        result = eval(expr, {"__builtins__": {}}, allowed)
        return _reply(f"{expr} = {result}", "got_it")
    except Exception as e:
        return f"That expression confused me: {e}"


def what_time(match):
    _think("checking the clock", seconds=0.6)
    now = datetime.datetime.now()
    return f"It's {now.strftime('%I:%M %p')} right now ({now.strftime('%A')})."


def what_date(match):
    _think("checking the calendar", seconds=0.6)
    now = datetime.datetime.now()
    return f"Today is {now.strftime('%A, %B %d, %Y')}."


def wikipedia(match):
    topic = match.group("topic").strip().rstrip("?.").strip()
    _think(f"looking up '{topic}' on Wikipedia", seconds=1.4)
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic.replace(' ', '_')}"
    r = _safe_get(url, headers={"User-Agent": "ArcturusJr/0.1"})
    if r is None or r.status_code != 200:
        return f"Couldn't find anything on Wikipedia for '{topic}'. Try a different phrasing?"
    data = r.json()
    extract = data.get("extract", "No summary available.")
    return _reply(f"{data.get('title', topic)}:\n\n{extract}", "got_it")


def define(match):
    word = match.group("word").strip()
    _think(f"finding the definition of '{word}'", seconds=1.2)
    r = _safe_get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{word}")
    if r is None or r.status_code != 200:
        return f"I couldn't find a definition for '{word}'. Maybe check your spelling?"
    data = r.json()
    try:
        meaning = data[0]["meanings"][0]
        definition = meaning["definitions"][0]["definition"]
        pos = meaning.get("partOfSpeech", "")
        return _reply(f"{word} ({pos}): {definition}", "got_it")
    except (KeyError, IndexError):
        return f"I got a response but couldn't parse it. Weird word, '{word}'."


def joke(match):
    _think("finding a good one", seconds=1.2)
    r = _safe_get("https://official-joke-api.appspot.com/random_joke")
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    data = r.json()
    return f"{data['setup']}\n\n...\n\n{data['punchline']}"


def cat_fact(match):
    _think("digging up a cat fact", seconds=1.0)
    r = _safe_get("https://catfact.ninja/fact")
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    return r.json().get("fact", "Cats are great.")


def dog_fact(match):
    _think("finding a dog fact", seconds=1.0)
    r = _safe_get("https://dogapi.dog/api/v2/facts")
    if r is None or r.status_code != 200:
        return "Dogs are the best. That's the fact. My API is down."
    try:
        return r.json()["data"][0]["attributes"]["body"]
    except (KeyError, IndexError):
        return "Dogs are good boys and girls. Official fact."


def quote(match):
    _think("finding a good quote", seconds=1.2)
    r = _safe_get("https://zenquotes.io/api/random")
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    try:
        q = r.json()[0]
        return f'"{q["q"]}"\n     — {q["a"]}'
    except (KeyError, IndexError):
        return _say("sorry_tool_failed")


def search(match):
    query = match.group("query").strip().rstrip("?.").strip()
    _think(f"searching for '{query}'", seconds=1.5)
    r = _safe_get(
        "https://api.duckduckgo.com/",
        params={"q": query, "format": "json", "no_html": 1}
    )
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    data = r.json()
    abstract = data.get("AbstractText") or data.get("Answer")
    if abstract:
        source = data.get("AbstractSource", "")
        return _reply(f"{abstract}" + (f"\n(via {source})" if source else ""), "got_it")
    related = data.get("RelatedTopics", [])
    if related:
        first = related[0]
        if isinstance(first, dict) and first.get("Text"):
            return _reply(first["Text"], "got_it")
    return f"DuckDuckGo didn't have an instant answer for '{query}'. Try rephrasing?"


def currency_convert(match):
    amount = float(match.group("amount"))
    src = match.group("src").upper()
    dst = match.group("dst").upper()
    _think("checking exchange rates", seconds=1.2)
    r = _safe_get(f"https://api.frankfurter.app/latest?amount={amount}&from={src}&to={dst}")
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    data = r.json()
    try:
        converted = data["rates"][dst]
        return _reply(f"{amount} {src} = {converted} {dst}", "got_it")
    except (KeyError, TypeError):
        return f"I couldn't convert {src} to {dst}. Are those real currency codes?"


def my_ip(match):
    _think("locating you", seconds=1.2)
    r = _safe_get("https://ipapi.co/json/")
    if r is None or r.status_code != 200:
        return _say("sorry_tool_failed")
    data = r.json()
    return _reply(
        f"You're at {data.get('ip', '?')}\n"
        f"Looks like you're in {data.get('city', '?')}, "
        f"{data.get('region', '?')}, {data.get('country_name', '?')}.",
        "got_it"
    )


def random_number(match):
    low = int(match.group("low"))
    high = int(match.group("high"))
    if low > high:
        low, high = high, low
    _think("rolling the dice", seconds=0.6)
    return f"Your random number is {random.randint(low, high)}."


def flip_coin(match):
    _think("flipping", seconds=1.0)
    return f"It's {random.choice(['Heads', 'Tails'])}!"


def roll_die(match):
    _think("rolling", seconds=1.0)
    return f"You rolled a {random.randint(1, 6)}."


# ============================================================
# THE ROUTER — the heart of this "agent"
# ============================================================
#
# This is the part that matters. Look at it. Really look.
#
# Every line below is a REGEX pattern mapped to a handler function.
# The "intelligence" of this agent is entirely in these patterns.
# If the user's query doesn't match any pattern EXACTLY, the agent
# fails. This is WHY we need LLMs — they replace this brittle
# pattern matching with actual understanding.

ROUTES = [
    # --- Small talk (order matters — check these before tools) ---
    (r"^(hi|hello|hey|yo|sup)\b", hello),
    (r"good (morning|afternoon|evening)", time_greeting),
    (r"how are you|how'?s it going|how are you doing|how do you do", how_are_you),
    (r"who are you|what('?s| is) your name|tell me about yourself|what are you", about_me),
    (r"who (made|created|built) you|who('?s| is) your (creator|maker)", who_made_you),
    (r"are you (a )?human|are you (an )?ai|are you real|are you a bot", are_you_human),
    (r"what can you do|help$|capabilities|what (are|do) you (know|capable)", capabilities),
    (r"\b(thanks|thank you|thx|appreciate it)\b", thanks),
    (r"you('?re| are) (awesome|cool|great|smart|amazing|the best)|i love you", compliment),
    (r"i('?m| am) bored|entertain me|surprise me", bored),
    (r"^(bye|goodbye|see (you|ya)|cya|exit|quit|stop)\b", goodbye),

    # --- Tools (order matters: MORE SPECIFIC patterns FIRST) ---

    # Weather — check BEFORE calculate. Also handle "temperature", "hot", "cold"
    (r"(?:what(?:'?s| is) )?(?:the )?(?:weather|temperature|temp|forecast|climate)\s+"
     r"(?:in |for |at |of |like in )?(?P<city>[\w\s,]+?)(?:\s+(?:today|now|right now))?(?:\?|\.|$)",
     weather),
    (r"(?:is it|how)\s+(?:hot|cold|warm|cool|raining|sunny|cloudy)\s+"
     r"(?:in |at |out in )?(?P<city>[\w\s,]+?)(?:\s+(?:today|now|right now))?(?:\?|\.|$)",
     weather),
    (r"weather\s+(?:in |for |at )?(?P<city>[\w\s,]+?)(?:\?|\.|$)", weather),

    # Calculate — require at least one DIGIT so "what's the temperature" doesn't match
    (r"(?:calculate|compute|evaluate|solve)\s+(?P<expr>.+)", calculate),

    (r"what(?:'?s| is) the time|what time is it|current time|time now", what_time),
    (r"what(?:'?s| is) (?:the |today'?s )?date|what day is it|today'?s date", what_date),

    (r"tell me about (?P<topic>.+?)(?:\?|$)", wikipedia),
    (r"who (?:is|was) (?P<topic>.+?)(?:\?|$)", wikipedia),
    (r"what (?:is|was) (?P<topic>[A-Z][\w\s]+?)(?:\?|$)", wikipedia),  # capitalized = proper noun

    (r"define (?P<word>\w+)|meaning of (?P<word2>\w+)|what does (?P<word3>\w+) mean", define),

    (r"tell me a joke|make me laugh|joke please|got a joke", joke),
    (r"cat fact|fact about cats|random cat", cat_fact),
    (r"dog fact|fact about dogs|random dog", dog_fact),
    (r"(?:give me |tell me )?(?:an? )?(?:inspirational |random )?quote", quote),

    (r"search (?:for |the web (?:for )?)?(?P<query>.+?)(?:\?|$)", search),
    (r"(?:look up|google) (?P<query>.+?)(?:\?|$)", search),

    (r"convert\s+(?P<amount>[\d.]+)\s*(?P<src>[A-Za-z]{3})\s+(?:to|in|into)\s+(?P<dst>[A-Za-z]{3})", currency_convert),
    (r"(?P<amount>[\d.]+)\s+(?P<src>[A-Za-z]{3})\s+(?:to|in|into)\s+(?P<dst>[A-Za-z]{3})", currency_convert),

    (r"(?:what(?:'?s| is) )?my ip|where am i", my_ip),

    (r"random number (?:between )?(?P<low>-?\d+)\s+(?:and|to|-)\s+(?P<high>-?\d+)", random_number),
    (r"flip a coin|heads or tails|coin flip", flip_coin),
    (r"roll a die|roll a dice|roll the dice", roll_die),
]


def _fix_define_groups(match):
    """The define pattern has three optional groups — normalize to 'word'."""
    word = match.group("word") or match.group("word2") or match.group("word3")
    class Fake:
        def group(self, name):
            return word if name == "word" else None
    return Fake()


def route(query):
    """
    THE ROUTING BRAIN.
    Try each regex in order. First match wins. Call the handler.
    Return the response, or None if nothing matched.
    """
    # Special case: pure math expression like "what is 2+2" or "2+2".
    # MUST contain a digit AND a math operator AND nothing but math chars.
    math_match = re.match(
        r"^\s*(?:what(?:'?s| is)?\s*)?(?P<expr>[-+*/().\d\s]+)\s*\??\s*$",
        query, re.IGNORECASE,
    )
    if (math_match
            and any(c.isdigit() for c in query)
            and any(c in query for c in "+-*/")):
        return calculate(math_match)

    for pattern, handler in ROUTES:
        if handler is None:  # skip placeholders
            continue
        m = re.search(pattern, query, re.IGNORECASE)
        if m:
            # Handle the multi-group 'define' pattern specially
            if handler is define and not m.groupdict().get("word"):
                return define(_fix_define_groups(m))
            return handler(m)
    return None


def handle_unknown(query):
    """When nothing matches — the WHOLE POINT of this demo."""
    responses = [
        f"Hmm, I don't know how to answer '{query}'. My regex didn't match anything.",
        f"Sorry, I don't understand '{query}'. I'm just regex — I need exact patterns.",
        f"I'm stumped by '{query}'. This is exactly where an LLM would help.",
    ]
    response = random.choice(responses)
    response += "\n\nType 'help' to see what I CAN do."
    response += "\n(And remember: an LLM-powered agent would handle this easily.)"
    return response


# ============================================================
# INTERACTIVE CHAT LOOP
# ============================================================

def chat():
    print("=" * 64)
    print(f"  {AGENT_NAME} — {AGENT_VERSION}")
    print("  (A fake agent with personality. No LLM inside.)")
    print("=" * 64)
    print()

    opener = (
        f"Hey! I'm {AGENT_NAME}. I look like an AI agent, "
        f"but I'm secretly just Python regex.\n"
        f"Type 'help' to see what I can do, or just chat with me.\n"
        f"Type 'bye' when you're done."
    )
    _type_out(f"{AGENT_NAME}: {opener}", delay=0.005)
    print()

    while True:
        try:
            user = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print(f"\n{AGENT_NAME}: Goodbye!")
            break

        if not user:
            continue

        response = route(user)
        if response is None:
            response = handle_unknown(user)
        elif response == GOODBYE:
            break

        print(f"{AGENT_NAME}: {response}")
        print()


# ============================================================
# DEMO MODE — run a bunch of canned queries for classroom demo
# ============================================================

DEMO_QUERIES = [
    "hello",
    "how are you?",
    "what can you do?",
    "what's the weather in Mumbai?",
    "what is 2**10 + 5",
    "tell me a joke",
    "cat fact",
    "define serendipity",
    "tell me about Alan Turing",
    "convert 100 USD to INR",
    "give me a quote",
    "flip a coin",
    "random number between 1 and 100",
    # Now the failures — these are the teaching moments
    "what's hot in Mumbai right now?",           # fails — regex can't infer 'hot' = weather
    "should I bring an umbrella tomorrow?",      # fails — needs reasoning
    "who's the guy that invented the computer?", # might fail — not strict pattern
    "find me a good joke, but make it clean",   # fails — extra constraint
    "bye",
]


def demo():
    """Run canned queries to show the agent in action (and in failure)."""
    print("=" * 64)
    print(f"  DEMO MODE — {AGENT_NAME}")
    print("  Watch what this agent CAN and CAN'T do.")
    print("=" * 64)
    print()

    for q in DEMO_QUERIES:
        print(f"You: {q}")
        response = route(q)
        if response is None:
            response = handle_unknown(q)
        elif response == GOODBYE:
            print(f"{AGENT_NAME}: See you!")
            break
        print(f"{AGENT_NAME}: {response}")
        print()
        time.sleep(0.3)


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "demo":
        demo()
    else:
        chat()


## `12_full_agent_ollama.py`

An adaptation of the full agent from earlier, configured to use a local LLM via Ollama instead of a cloud-based API, providing a private and offline alternative.

In [ ]:
"""
Session 3 - Demo 12: The Full Agent — running on LOCAL Ollama (gemma4:26b)
=============================================================================

This is the EXACT same agent as 10_full_agent.py. Same system prompt, same
tools, same parser, same loop, same tests. The ONLY thing that changed is
the LLM backend — instead of calling Google's Gemini API, we call a model
running locally via Ollama.

Why this matters:
  - No API key needed. No rate limits. No internet dependency.
  - Your data never leaves your machine.
  - Prove the point: an agent is (router + tools). The router can be
    ANY LLM. Cloud, local, open, closed — doesn't matter.

Setup:
  1. Install Ollama from https://ollama.com
  2. Pull the model:       ollama pull gemma4:26b
  3. Start Ollama server:  ollama serve   (usually runs in background)
  4. Install dependency:   pip install requests python-dotenv
  5. Run this file:        python 12_full_agent_ollama.py

Configuration (optional, via .env):
  OLLAMA_HOST=http://localhost:11434
  OLLAMA_MODEL=gemma4:26b
"""
import requests
import json
import re
import math
import os
import inspect
from dotenv import load_dotenv

# ============================================================
# Configuration
# ============================================================
load_dotenv()

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:26b")


def call_llm(prompt: str) -> str:
    """
    Send a prompt to local Ollama and return the text response.

    This is the ONLY function that differs from 10_full_agent.py.
    Everything else — tools, parser, loop — is identical.

    We use Ollama's /api/generate endpoint with format="json" so the model
    is forced to return valid JSON (local models can be less disciplined
    about output formatting than Gemini/Claude).
    """
    try:
        response = requests.post(
            f"{OLLAMA_HOST}/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,          # we want the full response at once
                "format": "json",         # force structured JSON output
                "options": {
                    "temperature": 0.1,   # low temp for consistent tool calls
                },
            },
            timeout=120,  # local inference can be slow on big models
        )
        response.raise_for_status()
        return response.json()["response"]
    except requests.ConnectionError:
        raise RuntimeError(
            f"Can't reach Ollama at {OLLAMA_HOST}. "
            f"Is it running? Try: ollama serve"
        )
    except requests.HTTPError as e:
        raise RuntimeError(
            f"Ollama API error: {e}. "
            f"Did you pull the model? Try: ollama pull {OLLAMA_MODEL}"
        )


# ============================================================
# System Prompt — identical to 10_full_agent.py
# ============================================================
system_prompt = """You are a helpful AI agent that can use tools to answer questions accurately.

You have access to the following tools:

1. calculate(expression: str) -> str
   Evaluate a mathematical expression using Python syntax.
   Examples: calculate("2**10"), calculate("math.sqrt(144)"), calculate("sum(math.exp(x) for x in [1,1,2,3,5,8])")

2. get_weather(city: str) -> str
   Get the current weather for a city.
   Examples: get_weather("Mumbai"), get_weather("London")

3. search_notes(query: str) -> str
   Search through user's notes for relevant information.
   Examples: search_notes("meeting"), search_notes("project ideas")

You must respond in ONE of these two JSON formats:

If you need to use a tool:
{"tool_name": "<name>", "tool_arguments": {"<arg_name>": "<value>"}}

If you have the final answer:
{"answer": "<your final answer>"}

CONCRETE EXAMPLES (follow this format EXACTLY):

User: What is the weather in Mumbai?
Response: {"tool_name": "get_weather", "tool_arguments": {"city": "Mumbai"}}

Tool Result: {"weather": {"temp": "32°C", "condition": "Humid"}}
Response: {"answer": "The weather in Mumbai is 32°C and humid."}

User: What is 5 plus 7?
Response: {"tool_name": "calculate", "tool_arguments": {"expression": "5 + 7"}}

Tool Result: {"result": "12"}
Response: {"answer": "5 plus 7 is 12."}

User: What is the capital of France?
Response: {"answer": "The capital of France is Paris."}

IMPORTANT RULES:
- The key for arguments is EXACTLY "tool_arguments" (not "tool_agents", not "args").
- tool_arguments is an OBJECT like {"city": "Mumbai"}, NEVER a raw string.
- Respond with ONLY the JSON. No other text. No markdown code fences.
- Use tools when you need real data or precise calculations.
- After receiving a tool result, either use another tool or provide your final answer.
- ALWAYS use the calculate tool for math — do NOT try to compute in your head.
"""


# ============================================================
# Tools — identical to 10_full_agent.py
# ============================================================

def calculate(expression: str) -> str:
    try:
        allowed = {
            "math": math, "abs": abs, "round": round, "pow": pow,
            "sum": sum, "min": min, "max": max, "range": range, "list": list,
        }
        result = eval(expression, {"__builtins__": {}}, allowed)
        return json.dumps({"result": str(result)})
    except Exception as e:
        return json.dumps({"error": f"Calculation failed: {str(e)}"})


def get_weather(city: str) -> str:
    weather_data = {
        "Mumbai": {"temp": "32°C", "condition": "Humid, Partly Cloudy", "humidity": "78%"},
        "Delhi": {"temp": "28°C", "condition": "Clear Sky", "humidity": "45%"},
        "London": {"temp": "15°C", "condition": "Rainy", "humidity": "85%"},
        "New York": {"temp": "22°C", "condition": "Sunny", "humidity": "55%"},
        "Tokyo": {"temp": "26°C", "condition": "Windy", "humidity": "60%"},
        "San Francisco": {"temp": "18°C", "condition": "Foggy", "humidity": "72%"},
    }
    if city in weather_data:
        return json.dumps({"weather": weather_data[city]})
    return json.dumps({"error": f"Weather data not available for {city}"})


def search_notes(query: str) -> str:
    notes = [
        {"title": "Meeting Agenda", "content": "Discuss Q3 targets, review agent architecture, plan MCP integration"},
        {"title": "Shopping List", "content": "Milk, eggs, bread, coffee, batteries"},
        {"title": "Project Ideas", "content": "Build a stock monitoring agent, voice-based assistant, browser automation tool"},
        {"title": "Travel Plans", "content": "Tokyo trip in December, need to book flights and hotel by November"},
        {"title": "Learning Notes", "content": "Finish transformer session, practice async Python, read MCP docs"},
    ]
    results = [
        n for n in notes
        if query.lower() in n["title"].lower() or query.lower() in n["content"].lower()
    ]
    if results:
        return json.dumps({"results": results})
    return json.dumps({"results": "No notes found matching your query"})


tools = {
    "calculate": calculate,
    "get_weather": get_weather,
    "search_notes": search_notes,
}


# ============================================================
# Response Parser — identical to 10_full_agent.py
# ============================================================

def extract_tool_args(parsed: dict, tool_name: str) -> dict:
    """
    Local models are sloppy about the exact argument key/shape. Gemma might
    emit 'tool_args', 'arguments', 'tool_agents' (typo!), or even put args
    at the top level of the response. Normalize all of these into a proper
    kwargs dict for the target tool.

    This is the kind of defensive plumbing you'll see everywhere when you
    work with smaller open models. Cloud models (Gemini, Claude) are more
    disciplined, but local models need babysitting.
    """
    # Possible key aliases the model might have invented
    CANDIDATE_KEYS = (
        "tool_arguments", "tool_args", "arguments", "args",
        "parameters", "params", "tool_agents",  # yes, Gemma said 'tool_agents'
        "input", "inputs",
    )

    raw = None
    for key in CANDIDATE_KEYS:
        if key in parsed:
            raw = parsed[key]
            break

    # Nothing matched — maybe the args are at the top level (sibling of tool_name)
    if raw is None:
        extras = {k: v for k, v in parsed.items()
                  if k not in ("tool_name", "answer", "name")}
        if extras:
            raw = extras

    # Already a dict — great, return as-is
    if isinstance(raw, dict):
        return raw

    # Got a raw string/number — wrap it using the tool's parameter name
    if raw is not None and tool_name in tools:
        sig = inspect.signature(tools[tool_name])
        params = [p for p in sig.parameters if p != "self"]
        if len(params) == 1:
            return {params[0]: raw}

    return {}


def parse_llm_response(text: str) -> dict:
    """Parse the LLM's response, handling common formatting issues"""
    text = text.strip()

    # Remove markdown code fences if present
    if text.startswith("```"):
        lines = text.split("\n")
        lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
        if text.startswith("json"):
            text = text[4:].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    raise ValueError(f"Could not parse LLM response: {text[:200]}")


# ============================================================
# The Agent Loop — identical to 10_full_agent.py
# ============================================================

def run_agent(user_query: str, max_iterations: int = 5, verbose: bool = True):
    """
    User query → LLM → [Tool call → Result → LLM]* → Final answer

    Same loop. Same pattern. Just a different model on a different machine.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"  User: {user_query}")
        print(f"{'='*60}")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ]

    for iteration in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {iteration + 1} ---")

        prompt = ""
        for msg in messages:
            if msg["role"] == "system":
                prompt += msg["content"] + "\n\n"
            elif msg["role"] == "user":
                prompt += f"User: {msg['content']}\n\n"
            elif msg["role"] == "assistant":
                prompt += f"Assistant: {msg['content']}\n\n"
            elif msg["role"] == "tool":
                prompt += f"Tool Result: {msg['content']}\n\n"

        response_text = call_llm(prompt)
        if verbose:
            print(f"LLM: {response_text.strip()}")

        try:
            parsed = parse_llm_response(response_text)
        except (ValueError, json.JSONDecodeError) as e:
            if verbose:
                print(f"Parse error: {e}")
                print("Asking LLM to retry...")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": "Please respond with valid JSON only. No markdown, no extra text."})
            continue

        if "answer" in parsed:
            if verbose:
                print(f"\n{'='*60}")
                print(f"  Agent Answer: {parsed['answer']}")
                print(f"{'='*60}")
            return parsed["answer"]

        if "tool_name" in parsed:
            tool_name = parsed["tool_name"]
            # Use the forgiving extractor instead of parsed["tool_arguments"]
            tool_args = extract_tool_args(parsed, tool_name)

            if verbose:
                print(f"→ Calling tool: {tool_name}({tool_args})")

            if tool_name not in tools:
                error_msg = json.dumps({"error": f"Unknown tool: {tool_name}. Available: {list(tools.keys())}"})
                if verbose:
                    print(f"→ Error: {error_msg}")
                messages.append({"role": "assistant", "content": response_text})
                messages.append({"role": "tool", "content": error_msg})
                continue

            try:
                tool_result = tools[tool_name](**tool_args)
            except TypeError as e:
                # Wrong / missing arguments — feed the error back so LLM retries
                error_msg = json.dumps({
                    "error": f"Bad arguments for {tool_name}: {e}. "
                             f"Expected format: {{'tool_name': '{tool_name}', "
                             f"'tool_arguments': {{...}}}}."
                })
                if verbose:
                    print(f"→ Error: {error_msg}")
                messages.append({"role": "assistant", "content": response_text})
                messages.append({"role": "tool", "content": error_msg})
                continue

            if verbose:
                print(f"→ Result: {tool_result}")

            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "tool", "content": tool_result})

    print("\nMax iterations reached. Agent could not complete the task.")
    return None


# ============================================================
# Connectivity check — fail fast with a clear message
# ============================================================

def _check_ollama():
    """Ping Ollama and verify the model is available before running tests."""
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        r.raise_for_status()
    except requests.RequestException:
        print(f"\nCan't reach Ollama at {OLLAMA_HOST}.")
        print("Start it with:  ollama serve")
        return False

    models = [m["name"] for m in r.json().get("models", [])]
    if not any(m.startswith(OLLAMA_MODEL.split(":")[0]) for m in models):
        print(f"\nModel '{OLLAMA_MODEL}' not found in Ollama.")
        print(f"Installed models: {', '.join(models) if models else '(none)'}")
        print(f"Pull it with:  ollama pull {OLLAMA_MODEL}")
        return False

    return True


# ============================================================
# Run it!
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print(f"  SESSION 3: FULL AGENT on LOCAL OLLAMA ({OLLAMA_MODEL})")
    print(f"  Same agent as 10_full_agent.py — different brain.")
    print("=" * 60)

    if not _check_ollama():
        raise SystemExit(1)

    # Test 1: Simple tool call
    print("\n\n>>> TEST 1: Simple weather query")
    run_agent("What is the weather in Mumbai?")

    # Test 2: Calculation
    print("\n\n>>> TEST 2: Math that LLMs get wrong")
    run_agent("What is 2 raised to the power of 10, plus the square root of 144?")

    # Test 3: Multi-step — needs multiple tool calls
    print("\n\n>>> TEST 3: Multi-step reasoning")
    run_agent(
        "Search my notes for travel plans, then check the weather in "
        "the city I'm planning to visit."
    )

    # Test 4: The classic — sum of exponentials of Fibonacci
    print("\n\n>>> TEST 4: Sum of exponentials of Fibonacci")
    run_agent(
        "Calculate the sum of exponential values (e^x) of the "
        "first 6 Fibonacci numbers: 1, 1, 2, 3, 5, 8"
    )

    # Test 5: Something that doesn't need tools
    print("\n\n>>> TEST 5: No tools needed")
    run_agent("What is the capital of France?")
